### Read Customer Data

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
df = spark.read.format("csv")\
    .option("inferSchema", True)\
    .option("header", True)\
    .load("/Volumes/ansh_catalog/bronze/bronze_volume/customers/")

In [0]:
display(df)

In [0]:
df.printSchema()

Change the name to upper case

In [0]:
df = df.withColumn("name", upper(col("name")))
display(df)

Extract the domain from the email

In [0]:
df = df.withColumn("domain", split(col("email"),"@")[1])
display(df)

Aggregate the data by domain

In [0]:
display(df.groupBy("domain")
        .agg(count(col("customer_id")).alias("total_customers"))
        .sort(col("total_customers").desc())
)

Add timestamp to each row

In [0]:
df = df.withColumn("processDate", current_timestamp())
display(df)

Write data into Silver

### Customers

In [0]:
from delta.tables import DeltaTable

In [0]:
if spark.catalog.tableExists("ansh_catalog.silver.customers_enriched"):
    dlt_obj = DeltaTable.forName(spark, "ansh_catalog.silver.customers_enriched")
    dlt_obj.alias("t").merge(
        df.alias("s"),
        "t.customer_id = s.customer_id")\
            .whenMatchedUpdateAll()\
            .whenNotMatchedInsertAll()\
            .execute()
else:
    df.write.format("delta")\
        .mode("append")\
        .saveAsTable("ansh_catalog.silver.customers_enriched")


### Products

In [0]:
df_products = spark.read.format("csv")\
    .option("inferSchema", True)\
    .option("header", True)\
    .load("/Volumes/ansh_catalog/bronze/bronze_volume/products/")
display(df_products)

In [0]:
df_products = df_products.withColumn("processedDate", current_timestamp())
display(df_products)


Average price per category

In [0]:
display(df_products.groupBy("category").agg(avg(col("price")).alias("avg_price")))

In [0]:
if spark.catalog.tableExists("ansh_catalog.silver.products_enriched"):
    dlt_obj = DeltaTable.forName(spark, "ansh_catalog.silver.products_enriched")
    dlt_obj.alias("t").merge(
        df_products.alias("s"),
        "t.product_id = s.product_id")\
            .whenMatchedUpdateAll()\
            .whenNotMatchedInsertAll()\
            .execute()
else:
    df_products.write.format("delta")\
        .mode("append")\
        .saveAsTable("ansh_catalog.silver.products_enriched")

In [0]:
%sql
SELECT * FROM ansh_catalog.silver.products_enriched

### STORES

In [0]:
df_stores = spark.read.format("csv")\
    .option("inferSchema", True)\
    .option("header", True)\
    .load("/Volumes/ansh_catalog/bronze/bronze_volume/stores/")
display(df_stores)

Remove the underscore for store name

In [0]:
df_stores = df_stores.withColumn("store_name", regexp_replace(col("store_name"), "_", ""))
display(df_stores)

In [0]:
df_stores = df_stores.withColumn("processedDate", current_timestamp())
display(df_stores)


In [0]:
if spark.catalog.tableExists("ansh_catalog.silver.stores_enriched"):
    dlt_obj = DeltaTable.forName(spark, "ansh_catalog.silver.stores_enriched")
    dlt_obj.alias("t").merge(
        df_stores.alias("s"),
        "t.store_id = s.store_id")\
            .whenMatchedUpdateAll()\
            .whenNotMatchedInsertAll()\
            .execute()
else:
    df_stores.write.format("delta")\
        .mode("append")\
        .saveAsTable("ansh_catalog.silver.stores_enriched")

In [0]:
%sql
SELECT * FROM ansh_catalog.silver.stores_enriched

### SALES

In [0]:
df_sales = spark.read.format("csv")\
    .option("inferSchema", True)\
    .option("header", True)\
    .load("/Volumes/ansh_catalog/bronze/bronze_volume/sales/")
display(df_sales)

In [0]:
df_sales = df_sales.withColumn("pricePerItem", round(col("total_amount")/col("quantity"), 2))\
    .withColumn("processedDate", current_timestamp())
display(df_sales)


In [0]:
if spark.catalog.tableExists("ansh_catalog.silver.sales_enriched"):
    dlt_obj = DeltaTable.forName(spark, "ansh_catalog.silver.sales_enriched")
    dlt_obj.alias("t").merge(
        df_sales.alias("s"),
        "t.sales_id = s.sales_id")\
            .whenMatchedUpdateAll()\
            .whenNotMatchedInsertAll()\
            .execute()
else:
    df_sales.write.format("delta")\
        .mode("append")\
        .saveAsTable("ansh_catalog.silver.sales_enriched")

In [0]:
%sql
SELECT * from ansh_catalog.silver.sales_enriched

In [0]:
display(spark.sql("select * from ansh_catalog.silver.sales_enriched"))

In [0]:
df_sql = spark.sql("select * from ansh_catalog.silver.products_enriched")
df_sql.createOrReplaceTempView("temp_products_enriched")

In [0]:
df_sql = spark.sql("""
          SELECT *, CASE
          WHEN category = 'Toys' THEN 'YES' ELSE 'NO' END AS is_toys
          FROM temp_products_enriched""")

In [0]:
display(df_sql)

### Pyspark UDF

In [0]:
def greet(p_input):
    return "Hello " + str(p_input)

udf_greet = udf(greet)
df_sql = df_sql.withColumn("is_toys", udf_greet(col("is_toys")))

display(df_sql)